In [0]:
df_cust=spark.table("etl.bronze.br_cust_info_crm")

In [0]:
%sql
select * from etl.bronze.br_cust_info_crm

In [0]:
from pyspark.sql import functions as F,Window


In [0]:
w = Window.partitionBy("cst_id")

df_val = df_cust.withColumn(
    "validation_errors",
    F.filter(
        F.array(
            F.when(
                F.col("cst_id").isNotNull() &
                (F.count("*").over(w) > 1),
                "DUPLICATE_CUSTOMER_ID"
            ),
            F.when(
                (F.col("cst_firstname") != F.trim(F.col("cst_firstname"))) |
                (F.col("cst_lastname") != F.trim(F.col("cst_lastname"))),
                "WHITESPACE_IN_NAME"
            ),
            F.when(
                F.col("cst_firstname").isNull() |
                F.col("cst_lastname").isNull() |
                F.col("cst_marital_status").isNull() |
                F.col("cst_gndr").isNull() |
                F.col("cst_create_date").isNull(),
                "NULL_VALUES"
            ),
            F.when(
                F.trim(F.col("cst_create_date")).isNull() |
                (F.trim(F.col("cst_create_date")) == ""),
                "MISSING_DATE"
            ),
            F.when(
                F.trim(F.col("cst_create_date")).isNotNull() &
                (F.trim(F.col("cst_create_date")) != "") &
                F.to_date(
                    F.trim(F.col("cst_create_date")),
                    "yyyy-MM-dd"
                ).isNull(),
                "INVALID_DATE_FORMAT"
            )
        ),
        lambda x: x.isNotNull()
    )
)

display(df_val.filter(F.size("validation_errors") > 0))  

In [0]:
##drop duplicates

df_dup=df_cust.dropDuplicates(["cst_id"])

display(df_dup.count())

In [0]:
##trim extra spaces

df_trim=df_dup.withColumn("cst_firstname",F.trim(F.col("cst_firstname"))).withColumn("cst_lastname",F.trim(F.col("cst_lastname")))

display(df_trim)


In [0]:
##fill nulls


df_fillna = df_trim.fillna({
    "cst_firstname": "Unknown_firstname",
    "cst_lastname": "Unknown_lastname",
    "cst_marital_status": "U",
    "cst_create_date": "1900-01-01",    
    "cst_gndr": "U"
})

display(df_fillna)

In [0]:
##identify abbreviation
# cst_gendr, cst_marital_status 

df_abbr=df_fillna.withColumn("cst_gndr",F.when(F.col("cst_gndr")=="M","Male").when(F.col("cst_gndr")=="F","Female").otherwise("Unknown")).withColumn("cst_marital_status",F.when(F.col("cst_marital_status")=="S","Single").when(F.col("cst_marital_status")=="M","Married").otherwise("Unknown"))

display(df_fillna)


In [0]:
#check datatype, format and missing values for date
# 1. Clean spaces
df = df_abbr.withColumn(
    "transaction_date_clean",
    F.trim(F.col("cst_create_date"))
)

#2 date format check
df = df.withColumn(
    "transaction_date_parsed",
    F.to_date(
        F.col("transaction_date_clean"),
        "yyyy-MM-dd"
    )
)

# 3. Missing dates
missing = df.filter(
    F.col("transaction_date_clean").isNull() |
    (F.col("transaction_date_clean") == "")
)
display(missing)
# 4. Invalid dates
invalid = df.filter(
    F.col("transaction_date_clean").isNotNull() &
    (F.col("transaction_date_clean") != "") &
    F.col("transaction_date_parsed").isNull()
)
display(invalid)
# 5. Valid dates
df_valid = df.filter(
    F.col("transaction_date_parsed").isNotNull()
)



In [0]:
#Standardize business key IDs to ensure tables can be joined correctly.

df = df_valid.withColumn(
    "cst_key",
    F.concat(
        F.lit("CST"),
        F.lit("_"),
        F.lit(""),
        F.col("cst_id")
    )
)
display(df)
#save to bronze table


In [0]:
df_valid.withColumnRenamed("cst_id","customer_id").withColumnRenamed("cst_key","customer_key").withColumnRenamed("cst_firstname","firstname").withColumnRenamed("cst_lastname","lastname").withColumnRenamed("cst_marital_status","marital_status").withColumnRenamed("cst_gndr","gender").withColumnRenamed("cst_create_date","create_date").withColumnRenamed("transaction_date_clean","transaction_date").withColumnRenamed("transaction_date_parsed","transaction_date_parsed").write.mode("overwrite").saveAsTable("etl.silver.sil_cust_info_crm")

In [0]:
%sql
select * from etl.silver.sil_cust_info_crm